In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import requests

from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
BASE_URL = "https://rest.uniprot.org/uniprotkb/search"

def download_uniprot_class(query, label_name, label_id, size=100):
    params = {
        "query": query,
        "format": "tsv",
        "fields": "accession,protein_name,gene_names,organism_name,length,sequence",
        "size": size
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        print("Error:", response.status_code)
        print(response.text)
        return None

    df = pd.read_csv(pd.io.common.StringIO(response.text), sep="\t")
    df["label_name"] = label_name
    df["label"] = label_id

    return df

In [ ]:
kinase_df = download_uniprot_class(
    query='reviewed:true AND protein kinase',
    label_name='kinase',
    label_id=0,
    size=100
)

kinase_df.head()

,Entry,Protein names,Gene Names,Organism,Length,Sequence,label_name,label
0,P19525,"Interferon-induced, double-stranded RNA-activa...",EIF2AK2 PKR PRKR,Homo sapiens (Human),551,MAGDLSAGFFMEELNTYRQKQGVVLKYQELPNSGPPHDRRFTFQVI...,kinase,0
1,Q16539,Mitogen-activated protein kinase 14 (MAP kinas...,MAPK14 CSBP CSBP1 CSBP2 CSPB1 MXI2 SAPK2A,Homo sapiens (Human),360,MSQERPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKT...,kinase,0
2,P17948,Vascular endothelial growth factor receptor 1 ...,FLT1 FLT FRT VEGFR1,Homo sapiens (Human),1338,MVSYWDTGVLLCALLSCLLLTGSSSGSKLKDPELSLKGTQHIMQAG...,kinase,0
3,P07948,Tyrosine-protein kinase Lyn (EC 2.7.10.2) (Lck...,LYN JTK8,Homo sapiens (Human),512,MGCIKSKGKDSLSDDGVDLKTQPVRNTERTIYVRDPTSNKQQRPVP...,kinase,0
4,O43318,Mitogen-activated protein kinase kinase kinase...,MAP3K7 TAK1,Homo sapiens (Human),606,MSTASAASSSSSSSAGEMIEAPSQVLNFEEIDYKEIEVEEVVGRGA...,kinase,0


In [ ]:
len(kinase_df)

100

In [ ]:
gpcr_df = download_uniprot_class(
    query='reviewed:true AND G-protein coupled receptor',
    label_name='GPCR',
    label_id=1,
    size=100
)

len(gpcr_df)

100

In [ ]:
gpcr_df.head()

,Entry,Protein names,Gene Names,Organism,Length,Sequence,label_name,label
0,Q8IZF2,Adhesion G protein-coupled receptor F5 (G-prot...,ADGRF5 GPR116 KIAA0758,Homo sapiens (Human),1346,MKSPRRTTLCLMFIVIYSSKAALNWNYESTIHPLSLHEHEPAGEEA...,GPCR,1
1,P47775,G-protein coupled receptor 12,GPR12,Homo sapiens (Human),334,MNEDLKVNLSGLPRDYLDAAAAENISAAVSSRVPAVEPEPELVVNP...,GPCR,1
2,Q9BZJ7,G-protein coupled receptor 62 (G-protein coupl...,GPR62,Homo sapiens (Human),368,MANSTGLNASEVAGSLGLILAAVVEVGALLGNGALLVVVLRTPGLR...,GPCR,1
3,P46089,G-protein coupled receptor 3 (ACCA orphan rece...,GPR3 ACCA,Homo sapiens (Human),330,MMWGAGSPLAWLSAGSGNVNVSSVGPAEGPTGPAAPLPSPKAWDVV...,GPCR,1
4,Q8IZF7,Putative adhesion G protein-coupled receptor F...,ADGRF2P ADGRF2 GPR111 PGR20,Homo sapiens (Human),708,MGLTAYGNRRVQPGELPFGANLTLIHTRAQPVICSKLLLTKRVSPI...,GPCR,1


In [ ]:
ion_df = download_uniprot_class(
    query='reviewed:true AND ion channel',
    label_name='ion_channel',
    label_id=2,
    size=100
)

len(ion_df)

ion_df.head()

,Entry,Protein names,Gene Names,Organism,Length,Sequence,label_name,label
0,Q7NDN8,Proton-gated ion channel (GLIC) (Ligand-gated ...,glvI glr4197,Gloeobacter violaceus (strain ATCC 29082 / PCC...,359,MFPTGWRPKLSESIAASRMLWQPMAAVAVVQIGLLWFSPPVWGQDM...,ion_channel,2
1,Q401N2,Ligand-gated cation channel ZACN (Ligand-gated...,ZACN L2 LGICZ LGICZ1 ZAC,Homo sapiens (Human),412,MMALWSLLHLTFLGFSITLLLVHGQGFQGTAAIWPSLFNVNLSKKV...,ion_channel,2
2,Q5H8A5,Ion channel POLLUX,POLLUX,Lotus japonicus (Lotus corniculatus var. japon...,917,MIPLPVAAANSNSNSNSNSNDEESPNLSTVIKPPLKKTKTLLPPPS...,ion_channel,2
3,Q5H8A6,Ion channel CASTOR,CASTOR,Lotus japonicus (Lotus corniculatus var. japon...,853,MSLDSEVSVSSSSGRDWFFPSPSFFRSSPSQYGRRFHTNSNTHSAP...,ion_channel,2
4,Q9NY37,Bile acid-sensitive ion channel (BASIC) (Acid-...,ASIC5 ACCN5,Homo sapiens (Human),505,MEQTEKSKVYAENGLLEKIKLCLSKKPLPSPTERKKFDHDFAISTS...,ion_channel,2


In [ ]:
combined_df = pd.concat(
    [kinase_df, gpcr_df, ion_df],
    ignore_index=True
)

combined_df = combined_df[["Sequence", "label", "label_name"]]

combined_df.head()

,Sequence,label,label_name
0,MAGDLSAGFFMEELNTYRQKQGVVLKYQELPNSGPPHDRRFTFQVI...,0,kinase
1,MSQERPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKT...,0,kinase
2,MVSYWDTGVLLCALLSCLLLTGSSSGSKLKDPELSLKGTQHIMQAG...,0,kinase
3,MGCIKSKGKDSLSDDGVDLKTQPVRNTERTIYVRDPTSNKQQRPVP...,0,kinase
4,MSTASAASSSSSSSAGEMIEAPSQVLNFEEIDYKEIEVEEVVGRGA...,0,kinase


In [ ]:
print(combined_df.shape)

combined_df["label_name"].value_counts()

(300, 3)


,count
label_name,
kinase,100
GPCR,100
ion_channel,100


In [ ]:
combined_df = combined_df.rename(
    columns={"Sequence": "sequence"}
)

train_df, test_df = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 240
Test size: 60


In [ ]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = {
    "train": train_dataset,
    "test": test_dataset
}

dataset

{'train': Dataset({
     features: ['sequence', 'label', 'label_name', '__index_level_0__'],
     num_rows: 240
 }),
 'test': Dataset({
     features: ['sequence', 'label', 'label_name', '__index_level_0__'],
     num_rows: 60
 })}

In [ ]:
model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["sequence"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./drug_target_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.


TrainOutput(global_step=180, training_loss=0.808944108751085, metrics={'train_runtime': 21.9987, 'train_samples_per_second': 32.729, 'train_steps_per_second': 8.182, 'total_flos': 16593674895360.0, 'train_loss': 0.808944108751085, 'epoch': 3.0})

In [ ]:
eval_results = trainer.evaluate()

eval_results

{'eval_loss': 0.6334683895111084,
 'eval_accuracy': 0.9,
 'eval_runtime': 0.5834,
 'eval_samples_per_second': 102.849,
 'eval_steps_per_second': 25.712,
 'epoch': 3.0}

In [ ]:
predictions_output = trainer.predict(tokenized_test)

logits = predictions_output.predictions
true_labels = predictions_output.label_ids

predicted_labels = np.argmax(logits, axis=-1)

In [ ]:
target_names = ["kinase", "GPCR", "ion_channel"]

report = classification_report(
    true_labels,
    predicted_labels,
    target_names=target_names
)

print(report)

              precision    recall  f1-score   support

      kinase       0.90      0.90      0.90        20
        GPCR       0.90      0.95      0.93        20
 ion_channel       0.89      0.85      0.87        20

    accuracy                           0.90        60
   macro avg       0.90      0.90      0.90        60
weighted avg       0.90      0.90      0.90        60



In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)

disp.plot()
plt.title("Confusion Matrix: Drug Target Class Prediction")
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    true_labels,
    predicted_labels,
    labels=[0, 1, 2]
)

metrics_df = pd.DataFrame({
    "class": target_names,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "support": support
})

metrics_df

,class,precision,recall,f1_score,support
0,kinase,0.900000,0.90,0.900000,20
1,GPCR,0.904762,0.95,0.926829,20
2,ion_channel,0.894737,0.85,0.871795,20


In [ ]:
plt.figure(figsize=(6,4))
plt.bar(metrics_df["class"], metrics_df["f1_score"])
plt.ylim(0, 1)
plt.ylabel("F1-score")
plt.title("F1-score per Drug Target Class")
plt.show()

In [ ]:
trainer.save_model("./final_drug_target_model")
tokenizer.save_pretrained("./final_drug_target_model")

('./final_drug_target_model/tokenizer_config.json',
 './final_drug_target_model/vocab.txt',
 './final_drug_target_model/added_tokens.json')

In [ ]:
label_map = {
    0: "kinase",
    1: "GPCR",
    2: "ion_channel"
}

def predict_drug_target_class(sequence):
    inputs = tokenizer(
        sequence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    outputs = model(**inputs)

    predicted_label = np.argmax(outputs.logits.detach().cpu().numpy(), axis=-1)[0]

    return label_map[predicted_label]

In [ ]:
test_sequence = "MELRVLLCWASLAAALEETLLNTKLETADLKWVTFPQVDGQWEELSGLDEEQHSVRTYEVCDGPGD"

prediction = predict_drug_target_class(test_sequence)

print("Predicted drug target class:", prediction)

Predicted drug target class: kinase


In [ ]:
trainer.save_model("./final_drug_target_model")
tokenizer.save_pretrained("./final_drug_target_model")

('./final_drug_target_model/tokenizer_config.json',
 './final_drug_target_model/vocab.txt',
 './final_drug_target_model/added_tokens.json')

In [ ]:
!zip -r final_drug_target_model.zip final_drug_target_model

  adding: final_drug_target_model/ (stored 0%)
  adding: final_drug_target_model/model.safetensors (deflated 7%)
  adding: final_drug_target_model/vocab.txt (deflated 6%)
  adding: final_drug_target_model/config.json (deflated 55%)
  adding: final_drug_target_model/training_args.bin (deflated 53%)
  adding: final_drug_target_model/tokenizer_config.json (deflated 76%)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import nbformat

notebook_path =/content/drive/MyDrive/Colab Notebooks/ESM2_Drug_Target_Class_Prediction.ipynb

nb = nbformat.read(notebook_path, as_version=4)

if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

nbformat.write(nb, notebook_path)

print("Notebook cleaned successfully!")

SyntaxError: invalid syntax (2961150431.py, line 3)